<div style="text-align:center;">
  <h1 size=10>
    <b>BIG DATA ANALYTICS PROJECT</b><br>
    <b>Baldness Classification</b>
  </h1>
</div>

<h2 style="text-align:center;">
Master's in Data Science and Advanced Analytics - NOVA IMS (25/26)
</h2>

**Group 26**
- Bárbara Franco (20250388)
- Catarina Mendinhas (20250422)
- Maria Miguel Fonseca (20250380)
- Rodrigo Santos (20250387)
- Rodrigo Teixeira (20250393)

**GitHub repository:**

<font color='#2f94d7' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>

- [1. Project Overview](#1)
- [2. Set Up & Import Libraries](#2)
- [3. Load Data](#3)
- [4. Data Exploration](#4)
    - [4.1 Independent Features](#4_1)
    - [4.2 Target Feature](#4_2)
    - [4.3 Relationship Between Text and Label](#4_3)
- [5. Data Split](#5)

# <font color='#2f94d7' size=6>**1. Project Overview**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

# <font color='#2f94d7' size=6>**2. Set Up & Import Libraries**</font> <a class="anchor" id="2"></a>

[Back to TOC](#toc)

In [1]:
!pip install pyspark

In [2]:
# IMPORT LIBRARIES
import os
import io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [3]:
# Install Java 17 (Required for Spark)
!sudo apt-get update
!sudo apt-get install -y openjdk-17-jdk-headless
!java -version

Get:1 https://download.docker.com/linux/ubuntu noble InRelease [48.5 kB]
Get:2 https://packages.cloud.google.com/apt cloud-sdk InRelease [1621 B]       
Get:3 https://cli.github.com/packages stable InRelease [3917 B]                
Hit:4 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble InRelease          
Get:5 https://packages.cloud.google.com/apt cloud-sdk/main all Packages [2000 kB]
Get:6 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://download.docker.com/linux/ubuntu noble/stable amd64 Packages [63.9 kB]
Get:8 https://cloud.archive.ubuntu.com/ubuntu noble InRelease [256 kB]         
Get:9 https://us-east-1.ec2.archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:10 https://packages.cloud.google.com/apt cloud-sdk/main amd64 Packages [4697 kB]
Get:11 https://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]    
Get:12 http://deb.wakemeops.com/wakemeops stable InRelease [50.5 kB]           
Get:13 https://clou

In [4]:
# Set JAVA_HOME and initialize Spark Session with specific configurations
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("BaldClassification")
    .config("spark.driver.memory", "12g") # RAM for the driven
    .config("spark.sql.files.maxPartitionBytes", "128m")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/11 17:04:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


In [5]:
# DEFINE DATA PATHS
BASE_PATH = "data/Data_Bald_People" 
SPLITS    = ["Train", "Validation", "Test"]
LABELS    = ["Bald", "NotBald"]

# <font color='#2f94d7' size=6>**3. Load Data**</font> <a class="anchor" id="3"></a>

[Back to TOC](#toc)

In [6]:
dfs = []

for split in SPLITS:
    for label in LABELS:
        folder = os.path.join(BASE_PATH, split, label)

        df_part = (
            spark.read.format("binaryFile")
            .option("pathGlobFilter", "*.jpg")   # muda para *.png se necessário
            .option("recursiveFileLookup", "false")
            .load(folder)
            .select(
                F.col("path"),
                F.col("content"),
                F.col("length").alias("file_size_bytes"),
            )
            .withColumn("label",          F.lit(label))
            .withColumn("label_index",    F.lit(1 if label == "Bald" else 0).cast("integer"))
            .withColumn("original_split", F.lit(split))
        )

        dfs.append(df_part)
        print(f"  Loaded {split}/{label}")

df_all = dfs[0]
for df in dfs[1:]:
    df_all = df_all.union(df)

df_all.cache()
print(f"\nTotal images loaded: {df_all.count():,}")

  Loaded Train/Bald


26/05/11 17:04:57 WARN SharedInMemoryCache: Evicting cached table partition metadata from memory due to size constraints (spark.sql.hive.filesourcePartitionFileCacheSize = 262144000 bytes). This may impact query planning performance.


  Loaded Train/NotBald
  Loaded Validation/Bald
  Loaded Validation/NotBald
  Loaded Test/Bald
  Loaded Test/NotBald



Total images loaded: 202,599


In [7]:
df_meta = df_all.drop("content")
df_meta.cache()
df_meta.createOrReplaceTempView("image_metadata")

df_meta.printSchema()

spark.sql("""
    SELECT
        original_split,
        label,
        COUNT(*)                                 AS n_images,
        ROUND(AVG(file_size_bytes) / 1024, 2)    AS avg_size_kb,
        ROUND(MIN(file_size_bytes) / 1024, 2)    AS min_size_kb,
        ROUND(MAX(file_size_bytes) / 1024, 2)    AS max_size_kb
    FROM image_metadata
    GROUP BY original_split, label
    ORDER BY original_split, label
""").show()

root
 |-- path: string (nullable = true)
 |-- file_size_bytes: long (nullable = true)
 |-- label: string (nullable = false)
 |-- label_index: integer (nullable = false)
 |-- original_split: string (nullable = false)



+--------------+-------+--------+-----------+-----------+-----------+
|original_split|  label|n_images|avg_size_kb|min_size_kb|max_size_kb|
+--------------+-------+--------+-----------+-----------+-----------+
|          Test|   Bald|     421|       6.08|       3.63|       9.79|
|          Test|NotBald|   19579|       6.84|        3.3|      14.42|
|         Train|   Bald|    3656|       6.04|       3.23|      11.45|
|         Train|NotBald|  156344|       6.81|       2.96|      18.87|
|    Validation|   Bald|     470|       6.02|       3.72|      12.66|
|    Validation|NotBald|   22129|       6.81|       2.81|      15.69|
+--------------+-------+--------+-----------+-----------+-----------+



# <font color='#2f94d7' size=6>**4. Class Imbalance**</font> <a class="anchor" id="4"></a>

[Back to TOC](#toc)